# bayesian a/b test

binomial conversion. pymc3 to estimate the lift posterior.


In [1]:
import numpy as np
import pymc3 as pm

# fake conversion data
n_a, c_a = 5000, 320
n_b, c_b = 5000, 360


In [2]:
with pm.Model() as m:
    p_a = pm.Beta('p_a', 1, 1)
    p_b = pm.Beta('p_b', 1, 1)
    obs_a = pm.Binomial('obs_a', n=n_a, p=p_a, observed=c_a)
    obs_b = pm.Binomial('obs_b', n=n_b, p=p_b, observed=c_b)
    lift = pm.Deterministic('lift', p_b - p_a)
    trace = pm.sample(2000, tune=1000, return_inferencedata=False)


In [3]:
import arviz as az
az.plot_posterior(trace, var_names=['lift'])


In [4]:
# probability b > a
lift_samples = trace['lift']
print('P(lift > 0):', (lift_samples > 0).mean())
print('mean lift:', lift_samples.mean())
print('95% hdi:', np.percentile(lift_samples, [2.5, 97.5]))


### key advantage of bayesian a/b: gives a direct probability of lift > 0.


In [5]:
# expected loss for each variant
risk_a = (lift_samples * (lift_samples > 0)).mean()
risk_b = -(lift_samples * (lift_samples < 0)).mean()
print('risk picking a:', risk_a)
print('risk picking b:', risk_b)


In [6]:
# compare with frequentist z-test
from statsmodels.stats.proportion import proportions_ztest
z, p = proportions_ztest([c_a, c_b], [n_a, n_b])
print('z:', z, 'p:', p)


### similar conclusion to z-test but more interpretable. will use this for blog post.


## todo
try larger context next.

In [ ]:
# tweak temperature
TEMPERATURE = 0.7